# Market Risk Analysis (VaR, CVaR, Monte Carlo)
This notebook analyzes and visualizes the Market Risk for a portfolio of stocks. It is fully self-contained so you can run all cells from top to bottom seamlessly!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import datetime
from scipy.stats import norm

# Set charting themes
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

### 1. Data Loading and Formatting

In [ ]:
def fetch_data(tickers, start_date, end_date=None):
    if not tickers:
        return pd.DataFrame()
    if end_date is None:
        end_date = datetime.date.today()
        
    # Download data
    data = yf.download(tickers, start=start_date, end=end_date)
    
    # Handle the new MultiIndex format and fallback efficiently
    if isinstance(data.columns, pd.MultiIndex):
        data = data['Adj Close'] if 'Adj Close' in data.columns.get_level_values(0) else data['Close']
    else:
        data = data['Adj Close'] if 'Adj Close' in data.columns else data['Close']
        
    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])
        
    return data.ffill().dropna()

def calculate_portfolio_returns(returns, weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    return returns.dot(weights)

# Define your portfolio and weights (must sum to 1)
portfolio_tickers = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ITC.NS']
weights = [0.3, 0.2, 0.2, 0.15, 0.15]

print(f"Fetching data for {portfolio_tickers}...")
price_data = fetch_data(portfolio_tickers, start_date='2020-01-01')
returns = price_data.pct_change().dropna()
portfolio_returns = calculate_portfolio_returns(returns, weights)

# Visualize Normalized Price History
plt.figure(figsize=(12, 6))
for ticker in price_data.columns:
    plt.plot(price_data.index, price_data[ticker] / price_data[ticker].iloc[0], label=ticker)

plt.title('Normalized Historical Prices of Portfolio Assets')
plt.ylabel('Normalized Price')
plt.legend()
plt.show()

### 2. Historical & Parametric Value at Risk (VaR)

In [ ]:
def calculate_historical_var(returns, confidence_level=0.95):
    return np.percentile(returns, 100 * (1 - confidence_level))

def calculate_parametric_var(returns, confidence_level=0.95):
    mean, std_dev = np.mean(returns), np.std(returns)
    return norm.ppf(1 - confidence_level, loc=mean, scale=std_dev)

def calculate_cvar(returns, confidence_level=0.95):
    var_threshold = calculate_historical_var(returns, confidence_level)
    tail_losses = returns[returns <= var_threshold]
    return tail_losses.mean() if len(tail_losses) > 0 else np.nan

confidence_level = 0.95

hist_var = calculate_historical_var(portfolio_returns, confidence_level)
param_var = calculate_parametric_var(portfolio_returns, confidence_level)
cvar = calculate_cvar(portfolio_returns, confidence_level)

print(f"Historical VaR (95%): {hist_var*100:.2f}% daily loss")
print(f"Parametric VaR (95%): {param_var*100:.2f}% daily loss")
print(f"CVaR (Expected Shortfall) (95%): {cvar*100:.2f}% daily loss")

# Visualize the Distribution of Portfolio Returns with Risk Thresholds
plt.figure(figsize=(12, 6))
sns.histplot(portfolio_returns, bins=60, kde=True, color='royalblue')
plt.axvline(hist_var, color='red', linestyle='--', linewidth=2, label=f'Historical VaR: {hist_var*100:.2f}%')
plt.axvline(param_var, color='orange', linestyle='-.', linewidth=2, label=f'Parametric VaR: {param_var*100:.2f}%')
plt.axvline(cvar, color='black', linestyle=':', linewidth=2, label=f'CVaR: {cvar*100:.2f}%')

plt.title('Distribution of Portfolio Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.legend()
plt.show()

### 3. Monte Carlo Simulation

In [ ]:
def simulate_monte_carlo_var(returns, confidence_level=0.95, num_simulations=100000):
    if len(returns) == 0: 
        return np.nan
    mean, std_dev = np.mean(returns), np.std(returns)
    simulated_returns = np.random.normal(mean, std_dev, num_simulations)
    return np.percentile(simulated_returns, 100 * (1 - confidence_level)), simulated_returns

np.random.seed(42)
simulations = 100000

mc_var, sim_returns = simulate_monte_carlo_var(portfolio_returns, confidence_level, simulations)
print(f"Monte Carlo VaR (95%): {mc_var*100:.2f}% daily loss")

plt.figure(figsize=(12, 6))
sns.histplot(sim_returns, bins=60, kde=True, color='forestgreen', alpha=0.6)
plt.axvline(mc_var, color='red', linestyle='--', linewidth=2, label=f'Monte Carlo VaR: {mc_var*100:.2f}%')
plt.title('Monte Carlo Simulated Return Distribution (100,000 paths)')
plt.xlabel('Simulated Daily Return')
plt.ylabel('Frequency')
plt.legend()
plt.show()

### 4. Stress Testing

In [ ]:
def run_stress_test(portfolio_returns, initial_investment, returns=None, weights=None, scenarios=None):
    historical_results = {}
    if isinstance(portfolio_returns, pd.Series) and not portfolio_returns.empty:
        worst_day_ret = portfolio_returns.min()
        historical_results['Historical Worst Day'] = worst_day_ret * initial_investment
        if len(portfolio_returns) >= 5:
            worst_week_ret = (portfolio_returns + 1).rolling(5).apply(np.prod, raw=True) - 1
            historical_results['Worst Week (5-day)'] = worst_week_ret.min() * initial_investment
        if len(portfolio_returns) >= 21:
            worst_month_ret = (portfolio_returns + 1).rolling(21).apply(np.prod, raw=True) - 1
            historical_results['Worst Month (21-day)'] = worst_month_ret.min() * initial_investment
        cumulative_returns = (1 + portfolio_returns).cumprod()
        peak = cumulative_returns.cummax()
        drawdown = (cumulative_returns - peak) / peak
        historical_results['Max Drawdown'] = drawdown.min() * initial_investment

    hypothetical_results = {}
    if returns is not None and weights is not None:
        tickers = returns.columns
        weighted_assets = sorted(zip(tickers, weights), key=lambda x: x[1], reverse=True)
        top_assets = weighted_assets[:min(3, len(weighted_assets))]
        for ticker, weight in top_assets:
            impact = initial_investment * weight * -0.20
            hypothetical_results[f'Isolation Shock: {ticker} drops 20%'] = impact
        worst_days = returns.min()
        extreme_crash_impact = np.sum(worst_days.values * weights) * initial_investment
        hypothetical_results['Concurrent Asset Worst Days'] = extreme_crash_impact
    else:
        if scenarios is None:
            scenarios = {'Black Monday (1987) Shock': -0.226, 'COVID-19 Crash (Daily)': -0.12}
        for scenario_name, shock_pct in scenarios.items():
            hypothetical_results[scenario_name] = initial_investment * shock_pct
    return historical_results, hypothetical_results

initial_investment = 1000000
print(f"Initial Portfolio Value: ₹{initial_investment:,.2f}\n")

# Run the upgraded stress tests on our downloaded portfolio
hist_res, hypo_res = run_stress_test(portfolio_returns, initial_investment, returns=returns, weights=weights)

print('--- Data-Driven Historical Worst Cases ---')
for scenario, impact in hist_res.items():
    print(f"{scenario}: impact of ₹{impact:,.2f}")

print('\n--- Asset-Specific Idiosyncratic Shocks ---')
for scenario, impact in hypo_res.items():
    print(f"{scenario}: impact of ₹{impact:,.2f}")
